# BatchNorm

Batch Normalization Overview

Batch Normalization (BatchNorm) is a technique used to stabilize and accelerate training of deep neural networks. It does so by normalizing the activations of a layer for each mini-batch of data. This helps address issues like vanishing/exploding gradients, and often speeds up convergence during training.

Here’s the general idea:

For each feature (i.e., for each input dimension in the batch), we calculate its mean and variance across the batch.
We normalize the feature by subtracting the batch mean and dividing by the batch standard deviation (which is the square root of the variance).
Scaling and shifting parameters (gamma and beta) are introduced to allow the model to learn the optimal normalized output (not just rely on a fixed normalization).
These parameters are learned during training.

The formula for BatchNorm for a feature $x$, in the batch is:

$$\hat{x}_i = \frac{ x_i - \mu }{ \sqrt{\sigma^2 + \epsilon} }$$

Where:
- $\mu$ is the mean of the feature across the mini-batch.
- $\sigma^2$ is the variance of the feature across the mini-batch.
- $\epsilon$ is a small constant added to the denominator for numerical stability.
- $\hat{x}_i$ is the normalized value.

Then, scaling and shifting are applied to the normalized output:

$$y_i = \gamma \hat{x}_i + \beta$$

Where:
- $\gamma$ is the scaling factor (learned during training).
- $\beta$ is the shifting factor (learned during training).

# What is running average

Running Averages: The Basics

A running average is an average that is updated incrementally as new data arrives, instead of recalculating the average from scratch every time. It is commonly used when you need to estimate the average of a sequence of values over time but cannot store all the data (for example, due to memory limitations or the continuous nature of the data).

Running Averages in Batch Normalization

In the context of Batch Normalization (BatchNorm), running averages are used to track the mean and variance of each feature (input dimension) during training across multiple mini-batches. These averages are maintained throughout training and are used during inference (testing), where we no longer have access to the full batch of data but still want to normalize the data in a consistent way.

How Running Averages Are Calculated

BatchNorm maintains two running averages:
- Running Mean (μ): The average of all the means computed across mini-batches.
- Running Variance (σ²): The average of all the variances computed across mini-batches.
At each mini-batch, the new running mean and running variance are updated using the following formulas:

`$$\mu_{\text{training}}^{t+1} = \alpha \cdot \mu_{\text{running}}^t + (1 - \alpha) \cdot \mu_{\text{batch}}$$

`$$\sigma_{\text{training}}^{2,t+1} = \alpha \cdot \sigma_{\text{running}}^{2,t} + (1 - \alpha) \cdot \sigma_{\text{batch}}^2$$

Where:
- $\mu_{\text{batch}}$ is the mean of the current mini-batch.
- $\sigma_{\text{batch}}^2$ is the variance of the current mini-batch.
- $\alpha$ is the smoothing factor (typically close to 1, e.g., 0.9 or 0.99). This controls how much weight is given to the new batch's statistics versus the previous running statistics.

Note: In PyTorch, the parameter `momentum` is the $\alpha$.

So, the running averages smooth out the statistics across mini-batches, making them more stable and reflective of the overall distribution of the data.

Key Points of Running Averages in BatchNorm
1. During Training: BatchNorm uses the statistics (mean and variance) of the current mini-batch for normalization. It also updates the running averages of the mean and variance with the new batch statistics.

2. During Inference: BatchNorm does not use the statistics from the current batch. Instead, it uses the running averages (mean and variance) accumulated during training.

3. Track Running Stats: In PyTorch, the track_running_stats flag controls whether running averages are tracked. If track_running_stats=True, BatchNorm tracks running statistics; otherwise, it uses the batch statistics directly.

In [10]:
import torch
import torch.nn as nn

# Define a simple model with BatchNorm
class SimpleBatchNormModel(nn.Module):
  def __init__(self, input_dim: int):
    super().__init__()
    # This layer will use BatchNorm (in this case, a single feature dimension)
    self.bn = nn.BatchNorm1d(input_dim, affine=True, track_running_stats=False)
    # track_running_stats=False for deterministic behavior

  def forward(self, x):
    return self.bn(x)

# Create a batch of data (e.g., 3 samples, 2 features)
x = torch.tensor([[1.0, 2.0],
                  [2.0, 3.0],
                  [3.0, 4.0]], dtype=torch.float32)

# Initialize the model
model = SimpleBatchNormModel(input_dim=2)

# Forward pass to calculate the BatchNorm
output = model(x)

# Display the intermediate steps for manual calculation
mean = x.mean(dim=0)
var = x.var(dim=0, unbiased=False)
epsilon = 1e-5

normalized = (x - mean) / torch.sqrt(var + epsilon)

# Gamma and Beta (they are initialized to 1 and 0 by default in BatchNorm)
gamma = model.bn.weight  # Gamma is learned during training
beta = model.bn.bias    # Beta is learned during training

# Apply scaling and shifting
batchnorm_output = gamma * normalized + beta

# Print results for manual comparison
print(f"Input Tensor:\n{x}")
print(f"Mean per feature:\n{mean}")
print(f"Variance per feature:\n{var}")
print(f"Normalized Tensor (without scaling and shifting):\n{normalized}")
print(f"Gamma (scaling):\n{gamma}")
print(f"Beta (shifting):\n{beta}")
print(f"BatchNorm Output (with scaling and shifting):\n{batchnorm_output}")

print(f"Output from nn.BatchNorm1d:\n{output}")

Input Tensor:
tensor([[1., 2.],
        [2., 3.],
        [3., 4.]])
Mean per feature:
tensor([2., 3.])
Variance per feature:
tensor([0.6667, 0.6667])
Normalized Tensor (without scaling and shifting):
tensor([[-1.2247, -1.2247],
        [ 0.0000,  0.0000],
        [ 1.2247,  1.2247]])
Gamma (scaling):
Parameter containing:
tensor([1., 1.], requires_grad=True)
Beta (shifting):
Parameter containing:
tensor([0., 0.], requires_grad=True)
BatchNorm Output (with scaling and shifting):
tensor([[-1.2247, -1.2247],
        [ 0.0000,  0.0000],
        [ 1.2247,  1.2247]], grad_fn=<AddBackward0>)
Output from nn.BatchNorm1d:
tensor([[-1.2247, -1.2247],
        [ 0.0000,  0.0000],
        [ 1.2247,  1.2247]], grad_fn=<NativeBatchNormBackward0>)


# LayerNorm

Overview of Layer Normalization

Layer Normalization (LayerNorm) is another technique used to normalize the activations of a layer in a neural network. Unlike Batch Normalization, which normalizes across the batch dimension (i.e., along the batch axis for each feature), LayerNorm normalizes each input feature across its own set of activations (i.e., along the feature axis for each data point in the batch). This makes it especially useful in sequence-based models (like RNNs or Transformers) where you don’t necessarily have a large batch size but still want to stabilize and accelerate training.

Formula for LayerNorm

The key idea behind LayerNorm is to normalize the inputs by computing the mean and variance for each individual sample (i.e., per data point, not across the batch). This is done as follows:

$$\hat{x}_i = \frac{ x_i - \mu }{ \sqrt{\sigma^2 + \epsilon} }$$

Where:
- $x_i$ is the value of each feature for a given data point.
- $\mu$ is the mean of the features for that data point.
- $\sigma^2$ is the variance of the features for that data point.
- $\epsilon$ is a small constant added to the denominator to ensure numerical stability.

After normalization, scaling and shifting parameters ($\gamma$ and $\beta$) are applied:

$$y_i = \gamma\hat{x}_i + \beta$$

Where:
- $\gamma$ is the scaling parameter (learnable).
- $\beta$ is the shifting parameter (learnable).

Key Differences from BatchNorm
- BatchNorm normalizes across the batch (across data points), while LayerNorm normalizes across the features (within a single data point).
- LayerNorm does not use the running statistics of the batch like BatchNorm. Instead, it computes the mean and variance per data point, which makes it particularly useful for recurrent models or models with variable batch sizes.

In [ ]:
import torch
import torch.nn as nn

# Define a simple model with LayerNorm
class SimpleLayerNormModel(nn.Module):
  def __init__(self, input_dim: int) -> torch.Tensor:
    super().__init__()
    # This layer will use LayerNorm (in this case, a single feature dimension)
    self.ln = nn.LayerNorm(input_dim, eps=1e-5, elementwise_affine=True)
    # elementwise_affine=True for gamma and beta

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    return self.ln(x)

# Create a batch of data (e.g., 3 samples, 2 features)
x = torch.tensor([[1.0, 2.0], 
                  [2.0, 3.0], 
                  [3.0, 4.0]], dtype=torch.float32)

# Initialize the model
model = SimpleLayerNormModel(input_dim=2)

# Forward pass to calculate the LayerNorm
output = model(x)

# Display the intermediate steps for manual calculation
mean = x.mean(dim=1, keepdim=True)
var = x.var(dim=1, unbiased=False, keepdim=True)
epsilon = 1e-5

normalized = (x - mean) / torch.sqrt(var + epsilon)

# Gamma and Beta (they are initialized to 1 and 0 by default in LayerNorm)
gamma = model.ln.weight  # Gamma is learned during training
beta = model.ln.bias    # Beta is learned during training

# Apply scaling and shifting
layernorm_output = gamma * normalized + beta

# Print results for manual comparison
print(f"Input Tensor:\n{x}")
print(f"Mean per sample (row):\n{mean}")
print(f"Variance per sample (row):\n{var}")
print(f"Normalized Tensor (without scaling and shifting):\n{normalized}")
print(f"Gamma (scaling):\n{gamma}")
print(f"Beta (shifting):\n{beta}")
print(f"LayerNorm Output (with scaling and shifting):\n{layernorm_output}")
print(f"Output from nn.LayerNorm:\n{output}")

Input Tensor:
tensor([[1., 2.],
        [2., 3.],
        [3., 4.]])
Mean per sample (row):
tensor([[1.5000],
        [2.5000],
        [3.5000]])
Variance per sample (row):
tensor([[0.2500],
        [0.2500],
        [0.2500]])
Gamma (scaling):
Parameter containing:
tensor([1., 1.], requires_grad=True)
Beta (shifting):
Parameter containing:
tensor([0., 0.], requires_grad=True)
Normalized Tensor (without scaling and shifting):
tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000],
        [-1.0000,  1.0000]])
LayerNorm Output (with scaling and shifting):
tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000],
        [-1.0000,  1.0000]], grad_fn=<AddBackward0>)
Output from nn.LayerNorm:
tensor([[-1.0000,  1.0000],
        [-1.0000,  1.0000],
        [-1.0000,  1.0000]], grad_fn=<NativeLayerNormBackward0>)
